# Hierarchical Clustering: Agglomerative & Divisive

## What This Notebook Covers
A complete, interview-ready walkthrough of **Hierarchical Clustering** — the family of algorithms that build nested cluster trees (dendrograms) without pre-specifying k. You will learn:
- The mathematical basis of agglomerative clustering and all four linkage criteria
- From-scratch implementation of Agglomerative clustering using only NumPy
- Sklearn implementation with dendrogram visualisation
- Linkage method comparison with silhouette scoring
- Key interview questions answered in depth

## Prerequisites
- Python 3.8+, NumPy, Pandas, Matplotlib, Seaborn, Scikit-learn, SciPy
- Basic familiarity with distance metrics and clustering concepts

## Dataset
We use the **Country Clustering** dataset (Country-data.csv) from Kaggle.

**Kaggle Link:** https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data

**Credits:** Rohan0301 on Kaggle. Contains socio-economic indicators (child mortality, income, GDPP, health expenditure etc.) for 167 countries — perfect for hierarchical clustering since natural groupings exist at multiple levels of granularity (continent → region → economic tier).

---
*GRADIENTTS ML Curriculum — Senior ML Engineer & MIT Educator Series*

In [ ]:
# ── CELL 2: ALL IMPORTS ──────────────────────────────────────────────────────

# Core numerical computing
import numpy as np

# Data manipulation
import pandas as pd

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Scipy — provides the dendrogram and linkage utilities
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster  # Hierarchical clustering tools
from scipy.spatial.distance import pdist, squareform               # Pairwise distance computation

# Sklearn — agglomerative clustering and evaluation
from sklearn.cluster import AgglomerativeClustering                # Sklearn hierarchical implementation
from sklearn.preprocessing import StandardScaler                   # Feature standardisation
from sklearn.metrics import silhouette_score                       # Cluster quality metric
from sklearn.decomposition import PCA                              # Dimensionality reduction for 2D viz

# Reproducibility
np.random.seed(42)

# Plot aesthetics
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('All imports loaded successfully.')

## Part 1: Theory Recap

**1. Bottom-Up (Agglomerative) Algorithm:**
Start with each of n points as its own cluster. At each step, merge the two **closest** clusters according to the chosen linkage criterion. Repeat until all points are in one cluster. The result is a binary tree called a **dendrogram**.

**2. Linkage Criteria — How We Measure Cluster Distance:**
- **Single linkage:** `d(A,B) = min_{a∈A, b∈B} dist(a,b)` — can chain (chaining effect).
- **Complete linkage:** `d(A,B) = max_{a∈A, b∈B} dist(a,b)` — compact, spherical clusters.
- **Average linkage (UPGMA):** `d(A,B) = (1/|A||B|) Σ dist(a,b)` — compromise, robust.
- **Ward linkage:** Minimises total within-cluster variance after merge. Tends to produce similarly sized clusters and is the default choice for most practitioners.

**3. Dendrogram Reading:**
The y-axis height at which two branches merge is the distance (or dissimilarity) at which those clusters were joined. Cut the dendrogram horizontally at a chosen height to extract k clusters.

**4. Complexity:**
Naïve agglomerative clustering is O(n³) time and O(n²) space. With optimisations (Prim's algorithm for single linkage), O(n² log n) is achievable. This is why hierarchical clustering struggles beyond ~10,000 points.

**5. Divisive Clustering:**
Top-down: start with one cluster containing all points, recursively split the cluster with the highest intra-cluster distance. Less common than agglomerative; computationally more expensive (O(2ⁿ) naïve).

## Load the Real-World Dataset
The Country Clustering dataset has 167 countries with 9 socio-economic features. Hierarchical clustering is ideal here because country groupings are naturally nested: individual countries → sub-regions → regions → development tiers.

In [ ]:
# ── CELL 4: LOAD DATASET ─────────────────────────────────────────────────────
# Download from: https://www.kaggle.com/datasets/rohan0301/unsupervised-learning-on-country-data
# Place 'Country-data.csv' in the same directory as this notebook.

try:
    df = pd.read_csv('Country-data.csv')
except FileNotFoundError:
    print('Country-data.csv not found. Download from the Kaggle link above.')
    print('Using a representative demo dataset...')
    np.random.seed(42)
    n = 167
    df = pd.DataFrame({
        'country': [f'Country_{i}' for i in range(n)],
        'child_mort': np.random.exponential(40, n).clip(1, 200),
        'exports': np.random.uniform(5, 85, n),
        'health': np.random.uniform(1, 17, n),
        'imports': np.random.uniform(5, 90, n),
        'income': np.random.exponential(15000, n).clip(500, 125000),
        'inflation': np.random.uniform(-5, 50, n),
        'life_expec': np.random.uniform(50, 85, n),
        'total_fer': np.random.uniform(1.1, 7.5, n),
        'gdpp': np.random.exponential(12000, n).clip(300, 105000),
    })

print('Shape:', df.shape)
print('\n--- HEAD ---')
print(df.head())
print('\n--- INFO ---')
df.info()
print('\n--- DESCRIBE ---')
print(df.describe())

print('\nFeature explanation:')
print('  child_mort : Child mortality per 1000 live births')
print('  exports    : % of GDP')
print('  health     : Health spending % of GDP')
print('  income     : Net per-capita income')
print('  life_expec : Life expectancy (years)')
print('  gdpp       : GDP per capita')
print('Task: Unsupervised — group countries by socio-economic similarity, no labels given.')

## Preprocessing
Country features span vastly different units — child mortality (0–200), GDP per capita (hundreds to 100,000+). Hierarchical clustering uses Euclidean distance by default, making StandardScaler essential. We also handle any null values.

In [ ]:
# ── CELL 5: PREPROCESSING ────────────────────────────────────────────────────

# Step 1: Separate country names (string identifier) from numeric features
country_names = df['country'].values      # Keep names for labelling the dendrogram
feature_cols = [c for c in df.columns if c != 'country']
X_raw = df[feature_cols].values

# Step 2: Check for and handle nulls — impute with column median (robust to outliers)
null_counts = df[feature_cols].isnull().sum()
print('Null counts before imputation:')
print(null_counts)

# Median imputation — better than mean for skewed socio-economic variables
for j in range(X_raw.shape[1]):
    col = X_raw[:, j]
    nan_mask = np.isnan(col)
    if nan_mask.any():
        col[nan_mask] = np.nanmedian(col)   # Replace NaN with column median

# Step 3: Standardise — zero mean, unit variance
# INTERVIEW NOTE: Ward linkage specifically minimises variance, making scaling DOUBLY important.
# Unscaled GDP (range: ~300–105000) would completely dominate the within-cluster variance term.
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'\nPost-scaling shape : {X.shape}')
print(f'Post-scaling mean  : {X.mean(axis=0).round(3)}')
print(f'Post-scaling std   : {X.std(axis=0).round(3)}')

# Limit dendrogram labels to first 50 countries for readability
X_viz = X[:50]
labels_viz = country_names[:50]

## Part 2: From Scratch Implementation
We implement **Agglomerative Hierarchical Clustering** from scratch using NumPy with **Ward linkage**. The implementation maintains a live distance matrix and a cluster membership map, merging the closest pair of clusters at each step.

In [ ]:
# ── CELL 7: AGGLOMERATIVE CLUSTERING FROM SCRATCH ─────────────────────────────

class AgglomerativeScratch:
    """
    Bottom-up agglomerative hierarchical clustering.
    Supports 'single', 'complete', and 'average' linkage.
    Implemented from first principles using NumPy only.

    Parameters
    ----------
    n_clusters : int — target number of clusters to extract
    linkage    : str — 'single' | 'complete' | 'average'
    """

    def __init__(self, n_clusters=3, linkage='complete'):
        self.n_clusters = n_clusters
        self.linkage = linkage
        self.labels_ = None
        self.merge_history_ = []   # Stores (cluster_a, cluster_b, distance) at each step

    def _compute_distance_matrix(self, X):
        """Compute the initial n×n Euclidean distance matrix."""
        n = X.shape[0]
        D = np.zeros((n, n))
        for i in range(n):
            for j in range(i + 1, n):
                # Euclidean distance between point i and point j
                dist = np.sqrt(np.sum((X[i] - X[j]) ** 2))
                D[i, j] = dist
                D[j, i] = dist   # Distance matrix is symmetric
        return D

    def _inter_cluster_distance(self, c_a, c_b, D):
        """
        Compute the distance between cluster c_a and cluster c_b
        using the selected linkage criterion.
        c_a, c_b are lists of point indices.
        """
        # Extract all pairwise distances between the two clusters
        pairwise = [D[i, j] for i in c_a for j in c_b]

        if self.linkage == 'single':
            # INTERVIEW NOTE: Single linkage — minimum distance.
            # Prone to 'chaining': one outlier can bridge two clusters.
            return min(pairwise)

        elif self.linkage == 'complete':
            # INTERVIEW NOTE: Complete linkage — maximum distance.
            # Produces compact, roughly equal-sized clusters.
            return max(pairwise)

        elif self.linkage == 'average':
            # INTERVIEW NOTE: Average linkage — mean of all pairwise distances.
            # UPGMA: Unweighted Pair Group Method with Arithmetic Mean.
            # Compromise between single and complete — most commonly used in biology.
            return sum(pairwise) / len(pairwise)

        else:
            raise ValueError(f'Unknown linkage: {self.linkage}')

    def fit(self, X):
        """
        Run agglomerative clustering on dataset X.
        Produces self.labels_ with cluster IDs in [0, n_clusters).
        """
        n = X.shape[0]

        # Precompute the full pairwise distance matrix — O(n²) space
        # INTERVIEW NOTE: This is the key bottleneck for large n.
        D = self._compute_distance_matrix(X)

        # Initialise: each point is its own cluster
        # clusters dict: cluster_id → list of point indices
        clusters = {i: [i] for i in range(n)}

        while len(clusters) > self.n_clusters:
            min_dist = np.inf
            merge_pair = (None, None)

            # Find the pair of clusters with minimum inter-cluster distance
            cluster_ids = list(clusters.keys())
            for i in range(len(cluster_ids)):
                for j in range(i + 1, len(cluster_ids)):
                    ci, cj = cluster_ids[i], cluster_ids[j]
                    # Compute linkage distance between cluster ci and cluster cj
                    d = self._inter_cluster_distance(clusters[ci], clusters[cj], D)
                    if d < min_dist:
                        min_dist = d
                        merge_pair = (ci, cj)

            # Merge the closest pair into the first cluster's slot
            ci, cj = merge_pair
            self.merge_history_.append((ci, cj, min_dist))
            clusters[ci] = clusters[ci] + clusters[cj]   # Combine point lists
            del clusters[cj]                               # Remove the absorbed cluster

        # Assign final integer labels [0, n_clusters)
        self.labels_ = np.empty(n, dtype=int)
        for label_id, (_, point_list) in enumerate(clusters.items()):
            for idx in point_list:
                self.labels_[idx] = label_id   # INTERVIEW NOTE: labels are 0-indexed

        return self

    def predict(self):
        """
        Return labels from the most recent fit().
        INTERVIEW NOTE: Like DBSCAN, agglomerative clustering is transductive.
        No centroids are stored, so predicting on new data requires re-fitting.
        """
        if self.labels_ is None:
            raise RuntimeError('Call fit() first.')
        return self.labels_


# ── Run on a subset for demo (O(n³) scratch implementation) ──────────────────
# Ward linkage not in scratch (requires special update formula); using 'complete'
X_sub = X[:40]   # 40 points for manageable O(n³) demo

agg_scratch = AgglomerativeScratch(n_clusters=3, linkage='complete')
agg_scratch.fit(X_sub)
scratch_labels = agg_scratch.predict()

print(f'From-Scratch Agglomerative Clustering (n=40, complete linkage, k=3):')
print(f'  Labels: {scratch_labels}')
unique, counts = np.unique(scratch_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} points')

In [ ]:
# ── CELL 8: EVALUATE SCRATCH MODEL ───────────────────────────────────────────

# Silhouette score evaluates cluster cohesion and separation
# Requires ≥ 2 clusters (which we have: k=3) and no noise labels
n_unique_scratch = len(set(scratch_labels))

if n_unique_scratch > 1:
    sil_scratch = silhouette_score(X_sub, scratch_labels)
    print(f'Silhouette Score (scratch, n=40, complete linkage): {sil_scratch:.4f}')
    print('Range: [−1, 1]. Values > 0.5 indicate well-separated clusters.')
else:
    print('Only one cluster found — cannot compute silhouette score.')

print('\nCluster sizes:')
unique, counts = np.unique(scratch_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} points')

print(f'\nFirst 5 merge events (cluster_a, cluster_b, distance):')
for step in agg_scratch.merge_history_[:5]:
    print(f'  Merge clusters {step[0]} + {step[1]} at distance {step[2]:.4f}')

## Part 3: Sklearn Implementation
Sklearn's `AgglomerativeClustering` uses efficient data structures (scipy's linkage under the hood) and supports all four linkage criteria, as well as connectivity constraints. We run it on the full dataset and compare all linkage methods head-to-head.

In [ ]:
# ── CELL 10: SKLEARN AGGLOMERATIVE CLUSTERING ─────────────────────────────────

linkage_methods = ['ward', 'complete', 'average', 'single']
results = []

for method in linkage_methods:
    # Ward linkage only works with Euclidean; others support arbitrary metrics
    agg = AgglomerativeClustering(n_clusters=3, linkage=method)
    sk_labels = agg.fit_predict(X)

    n_cl = len(set(sk_labels))
    sil = silhouette_score(X, sk_labels) if n_cl > 1 else np.nan

    results.append({
        'Linkage': method,
        'N Clusters': n_cl,
        'Silhouette': round(sil, 4)
    })

results_df = pd.DataFrame(results)
print('=== Linkage Method Comparison (k=3, full 167-country dataset) ===')
print(results_df.to_string(index=False))

# Run Ward for final labels (typically best for balanced, compact clusters)
best_agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
ward_labels = best_agg.fit_predict(X)

print(f'\nWard cluster sizes:')
unique, counts = np.unique(ward_labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  Cluster {u}: {c} countries')

print(f'\nScratch (complete, n=40): {sil_scratch:.4f}')
ward_sil = silhouette_score(X, ward_labels)
print(f'Sklearn  (ward, n=167)  : {ward_sil:.4f}')
print('\nDifference explained: larger dataset + Ward linkage optimises variance directly.')

## Visualisations
Two essential plots: (1) the **dendrogram** — the signature visual of hierarchical clustering — and (2) **PCA projection** of the Ward clusters to see how well-separated they are in 2D.

In [ ]:
# ── CELL 11: VISUALISATIONS ───────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Plot 1: Dendrogram (first 50 countries for readability) ──
ax1 = axes[0]

# Compute linkage matrix using scipy (Ward method)
Z = linkage(X_viz, method='ward', metric='euclidean')

# Plot dendrogram — y-axis shows merge distance, x-axis shows data points
plt.sca(ax1)
dend = dendrogram(
    Z,
    labels=labels_viz,
    leaf_rotation=90,       # Rotate x-axis labels for readability
    leaf_font_size=6,
    color_threshold=7.5,    # Horizontal cutoff — branches below this height are same colour
    ax=ax1
)
# Draw the cut threshold line — this is where we extract k clusters
ax1.axhline(y=7.5, color='red', linestyle='--', linewidth=1.5, label='Cut threshold (k=3)')
ax1.set_title('Dendrogram — Hierarchical Clustering\n(Ward Linkage, 50 Countries)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Country', fontsize=11)
ax1.set_ylabel('Merge Distance (Ward)', fontsize=11)
ax1.legend(fontsize=10)

# ── Plot 2: PCA projection of Ward clusters ──
ax2 = axes[1]

# Reduce to 2D using PCA for visualisation only (clustering was done in 9D)
pca = PCA(n_components=2, random_state=42)
X_2d = pca.fit_transform(X)
variance_explained = pca.explained_variance_ratio_

palette = sns.color_palette('Set2', n_colors=3)
for cluster_id in range(3):
    mask = ward_labels == cluster_id
    ax2.scatter(
        X_2d[mask, 0], X_2d[mask, 1],
        c=[palette[cluster_id]],
        label=f'Cluster {cluster_id} (n={mask.sum()})',
        s=70, alpha=0.85, edgecolors='white', linewidths=0.5
    )

ax2.set_title('Ward Agglomerative Clusters — PCA Projection\n(9D → 2D; clustering done in full space)', fontsize=13, fontweight='bold')
ax2.set_xlabel(f'PC1 ({variance_explained[0]*100:.1f}% variance)', fontsize=11)
ax2.set_ylabel(f'PC2 ({variance_explained[1]*100:.1f}% variance)', fontsize=11)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.savefig('hierarchical_clustering_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved as hierarchical_clustering_visualization.png')

## Part 4: Hyperparameter Experiments
Two key decisions in hierarchical clustering: **linkage method** and **number of clusters (k)**. We sweep both and use the silhouette score and dendrogram height gaps to guide the choice.

In [ ]:
# ── CELL 13: HYPERPARAMETER EXPERIMENTS ──────────────────────────────────────

# ── Experiment 1: Sweep k for Ward linkage ──
k_values = range(2, 9)
sil_scores_k = []

for k in k_values:
    lbl = AgglomerativeClustering(n_clusters=k, linkage='ward').fit_predict(X)
    sil = silhouette_score(X, lbl)
    sil_scores_k.append(sil)

# ── Experiment 2: Linkage vs k heat map (silhouette) ──
linkages = ['ward', 'complete', 'average', 'single']
k_range = [2, 3, 4, 5]
heatmap_data = np.zeros((len(linkages), len(k_range)))

for i, link in enumerate(linkages):
    for j, k in enumerate(k_range):
        lbl = AgglomerativeClustering(n_clusters=k, linkage=link).fit_predict(X)
        heatmap_data[i, j] = silhouette_score(X, lbl)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Silhouette vs k
ax1 = axes[0]
ax1.plot(list(k_values), sil_scores_k, 'o-', color='steelblue', linewidth=2, markersize=8)
best_k_idx = np.argmax(sil_scores_k)
ax1.axvline(x=list(k_values)[best_k_idx], color='red', linestyle='--', alpha=0.7,
            label=f'Best k={list(k_values)[best_k_idx]}')
ax1.set_title('Silhouette Score vs. Number of Clusters\n(Ward Linkage, Country Data)', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (k)', fontsize=11)
ax1.set_ylabel('Silhouette Score', fontsize=11)
ax1.legend()

# Silhouette heatmap: linkage × k
ax2 = axes[1]
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.3f',
    xticklabels=k_range,
    yticklabels=linkages,
    cmap='YlGnBu',
    ax=ax2,
    linewidths=0.5
)
ax2.set_title('Silhouette Score: Linkage × k\n(Higher = Better Separation)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (k)', fontsize=11)
ax2.set_ylabel('Linkage Method', fontsize=11)

plt.tight_layout()
plt.savefig('hierarchical_hyperparameter_experiments.png', dpi=150, bbox_inches='tight')
plt.show()
print('Hyperparameter figure saved.')
print(f'\nBest k (Ward): {list(k_values)[best_k_idx]}, Silhouette: {sil_scores_k[best_k_idx]:.4f}')

In [ ]:
# ── CELL 14: DENDROGRAM GAP ANALYSIS — automated k selection ─────────────────

# The largest gap between consecutive merge heights in the dendrogram
# indicates the most natural number of clusters.
Z_full = linkage(X, method='ward', metric='euclidean')

# Last 10 merge distances (largest merges — where k changes the most)
last_merges = Z_full[-10:, 2]  # Column 2 = merge distance
gaps = np.diff(last_merges[::-1])  # Differences between consecutive heights (descending)

print('Dendrogram Gap Analysis (Ward linkage, last 10 merges):')
print('Merge heights (descending):', np.round(last_merges[::-1], 3))
print('Gaps between levels       :', np.round(gaps, 3))
print(f'\nLargest gap at index {np.argmax(gaps)} → suggests k = {np.argmax(gaps) + 2}')
print('\nInterpretation: The largest gap means the biggest jump in merge cost —')
print('i.e., the algorithm had to bridge a large natural gap between cluster groups.')

## Part 5: Interview Corner

### Q: Why use Ward linkage over single/complete?
Ward linkage minimises the **increase in total within-cluster variance** after each merge. It tends to produce **compact, balanced clusters** of roughly equal size, which is desirable in most practical settings. Single linkage is prone to *chaining* (one intermediary point can merge very different groups). Complete linkage handles chaining but can break apart true clusters that have a few distant edge points. Average linkage is a compromise. **Ward is the practitioner default** unless there's a domain reason for another choice.

### Q: How do you choose k in hierarchical clustering without a validation set?
Three complementary approaches: (1) **Dendrogram inspection** — look for the largest vertical gap (biggest jump in merge height) and cut there. (2) **Silhouette sweep** — run AgglomerativeClustering for k=2…10 and pick k with the highest silhouette score. (3) **Domain knowledge** — if you know there are 3 income tiers in the data, use k=3.

### Q: Hierarchical vs K-Means — when to choose which?
| Property | Hierarchical | K-Means |
|---|---|---|
| Need to specify k? | Optional (dendrogram) | Yes |
| Cluster shape | Flexible (depends on linkage) | Convex only |
| Scalability | O(n² log n) — up to ~10k pts | O(nkI) — scalable |
| Deterministic? | Yes | No (random init) |
| Provides hierarchy? | Yes | No |
| Can undo merges? | No | Via re-run |

### Q: What is the dendrogram y-axis measuring exactly?
In Ward linkage, the y-axis shows the **increase in total within-cluster sum of squares** when two clusters merge. In complete linkage, it shows the **maximum pairwise distance** between the two clusters. In single linkage, it shows the **minimum pairwise distance**. Always know what your y-axis means — interviewers test this.

## Key Takeaways — 5 Bullets for Placement Interviews

1. **Hierarchical clustering produces a tree, not a partition:** The dendrogram lets you extract any number of clusters by choosing a cut height — you don't need to commit to k upfront.

2. **Ward linkage is the practitioner default:** It minimises within-cluster variance at each merge step, producing balanced, compact clusters — outperforms single/complete linkage in most real datasets.

3. **Complexity kills scalability:** Naïve implementation is O(n³) time and O(n²) space — hierarchical clustering is impractical above ~10,000 points without approximations (mini-batch, BIRCH, or graph-based methods).

4. **The algorithm is deterministic:** Unlike K-Means, hierarchical clustering has no random initialisation — running it twice on the same data always produces the same dendrogram.

5. **Largest dendrogram gap = natural k:** The biggest vertical distance between consecutive merge heights in the dendrogram reveals where the most natural cluster boundary lies — this is the standard method for choosing k without labels.